# Random Forest - Cross Validation (STROKE)

Notebook para executar Random Forest com StratifiedKFold sobre o dataset STROKE.
- Selecione qual dataset (raw ou normalizado) usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `rf_params`.
- Outputs: `model_reports/random_forest/cv/` contém `csv/`, `images`, `pdf`.
- O modelo será salvo em `databases/STROKE/common/models/random_forest/v1/rf_model_cf{N_SPLITS}.pkl`. 

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Use o dataset raw de STROKE por padrão; EDA pode gerar versões normalizadas se desejar
# Nome do dataset cru: healthcare_stroke_data.csv
DATASET_NAME = 'healthcare_stroke_data_processed_cru_final.csv'  # ou stroke_standard_scaled.csv / stroke_robust_scaled.csv / stroke_minmax_scaled.csv
# Preferir carregar do raw; se quiser usar versões processadas, ajuste a linha abaixo
DATASET_PATH = Path('../data/processed')/ DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Para STROKE normalmente a coluna alvo é 'stroke' (0/1)
TARGET_COLUMN = 'stroke' # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/random_forest/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/random_forest/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# Random Forest params (variável conforme solicitado)
rf_params = {
    "n_estimators": 650,
    "max_depth": 16,
    "min_samples_split": 8,
    "min_samples_leaf": 15,
    "max_features": 0.3,
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "criterion": "entropy",
    'n_jobs': 8
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid', 'stroke']

# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TARGET_COLUMN = 'stroke'  # ajustar se necessário
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42

In [2]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [3]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# df['y'] = df['stroke']
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)
def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in ['y', 'stroke', 'target', 'label', 'class']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, tenta converter diretamente (para STROKE, deve ser 0/1)
if not np.issubdtype(df['y'].dtype, np.number):
    # factorize para inteiros 0..k-1
    df['y'], uniques = pd.factorize(df['y'])
    # salva versão discretizada para inspeção
    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# reorganiza para garantir que y seja a última coluna
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\healthcare_stroke_data_processed_cru_final_raw.csv
Dataset shape: (4259, 12)


In [4]:
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,y
0,31112,Male,80.0,No,Yes,Yes,Private,Rural,105.92,32.5,never smoked,1
1,53882,Male,74.0,Yes,Yes,Yes,Private,Rural,70.09,27.4,never smoked,1
2,10434,Female,69.0,No,No,No,Private,Urban,94.39,22.8,never smoked,1
3,60491,Female,78.0,No,No,Yes,Private,Urban,58.57,24.2,Unknown,1
4,12109,Female,81.0,Yes,No,Yes,Private,Rural,80.43,29.7,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
4254,14180,Female,13.0,No,No,No,children,Rural,103.08,18.6,Unknown,0
4255,44873,Female,81.0,No,No,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
4256,19723,Female,35.0,No,No,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
4257,37544,Male,51.0,No,No,Yes,Private,Rural,166.29,25.6,formerly smoked,0


In [5]:
# ============================ 8/1/1 HOLDOUT + OPTUNA (RF) OTIMIZANDO AUROC ============================
# Objetivo: manter AUROC como métrica de otimização (Optuna) e AJUSTAR THRESHOLD no holdout (fold 9)
# para evitar o colapso TP=0 (muito comum em Stroke ~3% positivos).
#
# Saídas:
#   cv_811_inner_folds.csv
#   cv_811_threshold_selection.csv         (NOVO)  -> tabela completa de thresholds no fold 9
#   cv_811_threshold_choices.csv           (NOVO)  -> top thresholds "interessantes" por critério
#   cv_811_validation.csv                  -> métricas no fold 9: threshold_0.5 + thresholds escolhidos
#   cv_811_test.csv                        -> métricas no fold 10: threshold_0.5 + thresholds escolhidos
#   cv_811_results_long.csv                -> tudo consolidado
#
# Importante: o fold 10 (teste) NÃO é usado para escolher threshold (sem vazamento).
# ================================================================================================

%pip install optuna

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.ensemble import RandomForestClassifier

# -------------------- CONFIG --------------------
RANDOM_STATE      = 42
N_TRIALS          = 30
N_SPLITS_OUTER    = 10
N_SPLITS_INNER    = 8
THRESHOLD_DEFAULT = 0.5
EPS              = 1e-12

# Thresholds "interessantes" (grid fixo + quantis dos scores do fold 9)
GRID_LINEAR = np.linspace(0.01, 0.50, 100)   # Stroke: probs costumam ficar < 0.5
GRID_FINE   = np.array([0.01,0.02,0.03,0.05,0.07,0.10,0.12,0.15,0.18,0.20,0.25,0.30,0.35,0.40,0.45,0.50])

# Restrições "interessantes" para saúde (você pode mudar números depois, sem tocar no teste):
FPR_CAPS = [0.01, 0.02, 0.03, 0.05]   # 1%, 2%, 3%, 5% de falsos positivos entre negativos
REC_TARGETS = [0.30, 0.40, 0.50]      # metas de recall (sensibilidade)

# -----------------------------------------------
# 1) Separa X,y sem vazamento
# -----------------------------------------------
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
id_cols = [c for c in df.columns if c.lower() in exclude_lower]
id_cols = [c for c in id_cols if c != 'y']
FEATURES = [c for c in df.columns if c not in id_cols + ['y']]

X_raw = df[FEATURES].copy()
y     = df['y'].copy()

num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in FEATURES if c not in num_cols]

num_pipe = Pipeline(steps=[("imp", SimpleImputer(strategy="median"))])
cat_pipe = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-1
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

# -----------------------------------------------
# 2) Define 8/1/1 holdout (fold 9 validação, fold 10 teste)
# -----------------------------------------------
outer_kf = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_raw, y))

(_, test_indices) = fold_indices[-1]   # fold 10 -> teste (holdout)
(_, val_indices)  = fold_indices[-2]   # fold 9  -> validação (holdout)

all_idx = np.arange(len(X_raw))
mask = np.ones(len(X_raw), dtype=bool)
mask[val_indices]  = False
mask[test_indices] = False
train8_idx = all_idx[mask]             # treino = 8 folds

X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val_indices], y.iloc[val_indices]
X_test1,  y_test1  = X_raw.iloc[test_indices], y.iloc[test_indices]

# -----------------------------------------------
# 3) OPTUNA (CV interna 8 folds) otimizando AUROC
# -----------------------------------------------
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)

def build_model(trial):
    n_estimators      = trial.suggest_int("n_estimators", 200, 1000, step=50)
    max_depth_choice  = trial.suggest_categorical("max_depth_choice", ["None", "int"])
    max_depth         = None if max_depth_choice == "None" else trial.suggest_int("max_depth", 3, 50, step=1)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20, step=1)
    min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 20, step=1)
    bootstrap         = trial.suggest_categorical("bootstrap", [True, False])
    class_weight      = trial.suggest_categorical("class_weight", [None, "balanced"])
    max_features_mode = trial.suggest_categorical("max_features_mode", ["sqrt", "log2", "float"])
    if max_features_mode == "float":
        max_features = trial.suggest_float("max_features_float", 0.2, 1.0)
    else:
        max_features = max_features_mode

    max_samples = None
    if bootstrap:
        max_samples = trial.suggest_float("max_samples", 0.5, 1.0)

    rf_params = dict(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    if bootstrap and (max_samples is not None):
        rf_params["max_samples"] = max_samples

    return Pipeline(steps=[
        ("prep", preprocess),
        ("model", RandomForestClassifier(**rf_params))
    ])

def _get_pos_proba(clf, X):
    proba = clf.predict_proba(X)
    if isinstance(proba, np.ndarray) and proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]
    return np.squeeze(proba)

def objective(trial):
    clf = build_model(trial)
    aucs = []
    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]
        clf.fit(X_tr, y_tr)
        p_pos = _get_pos_proba(clf, X_va)
        auc = roc_auc_score(y_va, p_pos)
        aucs.append(auc)
        trial.report(float(auc), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

study = optuna.create_study(
    study_name="rf_optuna_8fold_search_catnum",
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=2),
)
study.optimize(objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

best = study.best_trial
bp = dict(best.params)
if bp.get("max_depth_choice") == "None":
    bp["max_depth"] = None
bp["max_features"] = bp.get("max_features_float", bp.get("max_features_mode"))
for k in ["max_depth_choice", "max_features_mode", "max_features_float"]:
    bp.pop(k, None)
if not bp.get("bootstrap", True):
    bp.pop("max_samples", None)
bp["n_jobs"] = -1
bp["random_state"] = RANDOM_STATE
rf_best_params = bp

print("Melhor AUROC (CV interna 8 folds):", best.value)
print("Melhores hiperparâmetros:", rf_best_params)

# -----------------------------------------------
# 4) Funções de métricas + seleção de threshold (fold 9)
# -----------------------------------------------
def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    y_true = np.asarray(y_true).astype(int)
    mask0 = (y_true == 0)
    mask1 = ~mask0
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def _confusion(y_true, proba, thr):
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)
    y_pred = (proba >= thr).astype(int)
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    return TP, FP, TN, FN

def all_metrics_table(y_true, proba, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)

    # AUROC/AUPRC não dependem de threshold
    auc   = float(roc_auc_score(y_true, proba)) if len(np.unique(y_true)) == 2 else np.nan
    auprc = float(average_precision_score(y_true, proba)) if len(np.unique(y_true)) == 2 else np.nan

    TP, FP, TN, FN = _confusion(y_true, proba, threshold)
    total = TP + TN + FP + FN

    ce0, ce1 = cross_entropy_per_class(y_true, proba, eps=eps)

    y_pred = (proba >= threshold).astype(int)
    acc  = float(accuracy_score(y_true, y_pred))
    f1   = float(f1_score(y_true, y_pred, zero_division=0))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec  = float(recall_score(y_true, y_pred, zero_division=0))

    tnr = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    fpr = FP / (FP + TN) if (FP + TN) > 0 else np.nan
    tpr = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    bal_acc = float(np.nanmean([tnr, tpr]))
    youden  = float(tpr - fpr) if (not np.isnan(tpr) and not np.isnan(fpr)) else np.nan

    n0 = int(np.sum(y_true == 0))
    n1 = int(np.sum(y_true == 1))
    prop_c0 = n0 / total if total > 0 else np.nan
    prop_c1 = n1 / total if total > 0 else np.nan

    return {
        "Threshold": float(threshold),
        "Acurácia": acc,
        "AUROC": auc,
        "AUPRC": auprc,
        "Cross-Entropy C0": ce0,
        "Proporção C0": prop_c0,
        "Cross-Entropy C1": ce1,
        "Proporção C1": prop_c1,
        "F1 Score": f1,
        "Balanced Accuracy": bal_acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity(TNR)": float(tnr) if tnr is not np.nan else np.nan,
        "FPR": float(fpr) if fpr is not np.nan else np.nan,
        "YoudenJ": youden,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": (TP/total if total else np.nan),
        "FP(%)": (FP/total if total else np.nan),
        "TN(%)": (TN/total if total else np.nan),
        "FN(%)": (FN/total if total else np.nan),
    }

def evaluate_threshold_grid(y_true, proba, thresholds):
    rows = []
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)

    for thr in thresholds:
        TP, FP, TN, FN = _confusion(y_true, proba, thr)

        prec = TP/(TP+FP) if (TP+FP) > 0 else 0.0
        rec  = TP/(TP+FN) if (TP+FN) > 0 else 0.0
        f1   = (2*prec*rec)/(prec+rec) if (prec+rec) > 0 else 0.0

        tnr = TN/(TN+FP) if (TN+FP) > 0 else 0.0
        fpr = FP/(FP+TN) if (FP+TN) > 0 else 0.0
        youden = rec - fpr

        rows.append({
            "threshold": float(thr),
            "precision": float(prec),
            "recall": float(rec),
            "f1": float(f1),
            "tnr": float(tnr),
            "fpr": float(fpr),
            "youden": float(youden),
            "TP": TP, "FP": FP, "TN": TN, "FN": FN
        })

    df_thr = pd.DataFrame(rows)
    # ordenações úteis
    df_thr["score_f1"] = df_thr["f1"]
    df_thr["score_youden"] = df_thr["youden"]
    return df_thr.sort_values(["score_f1","score_youden"], ascending=False).reset_index(drop=True)

def pick_interesting_thresholds(df_thr):
    """
    Retorna um dicionário com thresholds "interessantes" (para reportar e comparar):
      - best_f1
      - best_youden
      - best_recall_fpr<=X (para X em FPR_CAPS)
      - min_thr_recall>=Y (para Y em REC_TARGETS) com menor FPR (e depois melhor F1)
    """
    choices = {}

    # (1) Max F1
    r = df_thr.sort_values(["f1","youden"], ascending=False).iloc[0]
    choices["best_f1"] = float(r["threshold"])

    # (2) Max Youden
    r = df_thr.sort_values(["youden","f1"], ascending=False).iloc[0]
    choices["best_youden"] = float(r["threshold"])

    # (3) Melhor Recall sob teto de FPR
    for cap in FPR_CAPS:
        sub = df_thr[df_thr["fpr"] <= cap].copy()
        if len(sub) == 0:
            continue
        r = sub.sort_values(["recall","f1"], ascending=False).iloc[0]
        choices[f"best_recall_fpr<={cap:.2f}"] = float(r["threshold"])

    # (4) Menor threshold que atinge recall>=target (prioriza recall; depois menor FPR; depois melhor F1)
    for target in REC_TARGETS:
        sub = df_thr[df_thr["recall"] >= target].copy()
        if len(sub) == 0:
            continue
        r = sub.sort_values(["fpr","threshold","f1"], ascending=[True, True, False]).iloc[0]
        choices[f"min_thr_recall>={target:.2f}_lowFPR"] = float(r["threshold"])

    return choices

# -----------------------------------------------
# 5) (1) Reavalia inner CV (8 folds) com threshold 0.5 (apenas para referência)
# -----------------------------------------------
inner_fold_rows = []
inner_clf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(**rf_best_params))
])

for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]
    inner_clf.fit(X_tr, y_tr)
    p_pos = _get_pos_proba(inner_clf, X_va)
    metrics = all_metrics_table(y_va, p_pos, threshold=THRESHOLD_DEFAULT, eps=EPS)
    inner_fold_rows.append({"etapa":"inner_cv","fold":fold_id,"modo":"threshold_0.5", **metrics})

df_inner = pd.DataFrame(inner_fold_rows)
df_inner.to_csv("cv_811_inner_folds.csv", index=False)

# -----------------------------------------------
# 6) (2) Validação holdout (fold 9): escolhe thresholds "interessantes"
# -----------------------------------------------
clf_val = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(**rf_best_params))
])
clf_val.fit(X_train8, y_train8)
p_val = _get_pos_proba(clf_val, X_val1)

# Monta thresholds: grid fixo + quantis dos scores (para capturar a “escala real” das probabilidades)
quantiles = np.quantile(p_val, [0.50,0.60,0.70,0.80,0.85,0.90,0.92,0.95,0.97,0.99])
thresholds = np.unique(np.clip(np.concatenate([GRID_LINEAR, GRID_FINE, quantiles]), 0.0, 1.0))
thresholds = thresholds[(thresholds > 0) & (thresholds <= 1)]

df_thr = evaluate_threshold_grid(y_val1, p_val, thresholds)
df_thr.to_csv("cv_811_threshold_selection.csv", index=False)

choices = pick_interesting_thresholds(df_thr)
df_choices = pd.DataFrame([{"criterio": k, "threshold": v} for k,v in choices.items()]).sort_values("criterio")
df_choices.to_csv("cv_811_threshold_choices.csv", index=False)

print("\n===== Thresholds 'interessantes' escolhidos no fold 9 =====")
print(df_choices.to_string(index=False))

# Monta linhas de validação: threshold 0.5 + cada threshold interessante
val_rows = []
val_rows.append({"etapa":"validacao","fold":9,"modo":"threshold_0.5", **all_metrics_table(y_val1, p_val, THRESHOLD_DEFAULT, EPS)})

for name, thr in choices.items():
    val_rows.append({"etapa":"validacao","fold":9,"modo":name, **all_metrics_table(y_val1, p_val, thr, EPS)})

df_val = pd.DataFrame(val_rows)
df_val.to_csv("cv_811_validation.csv", index=False)

# -----------------------------------------------
# 7) (3) Treino final (8+1) e teste holdout (fold 10): aplica os MESMOS thresholds
# -----------------------------------------------
X_train9 = pd.concat([X_train8, X_val1])
y_train9 = pd.concat([y_train8, y_val1])

clf_final = Pipeline(steps=[
    ("prep", preprocess),
    ("model", RandomForestClassifier(**rf_best_params))
])
clf_final.fit(X_train9, y_train9)
p_test = _get_pos_proba(clf_final, X_test1)

test_rows = []
test_rows.append({"etapa":"teste","fold":10,"modo":"threshold_0.5", **all_metrics_table(y_test1, p_test, THRESHOLD_DEFAULT, EPS)})

for name, thr in choices.items():
    test_rows.append({"etapa":"teste","fold":10,"modo":name, **all_metrics_table(y_test1, p_test, thr, EPS)})

df_test = pd.DataFrame(test_rows)
df_test.to_csv("cv_811_test.csv", index=False)

# -----------------------------------------------
# 8) (4) Consolida tudo
# -----------------------------------------------
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)
df_all.to_csv("cv_811_results_long.csv", index=False)

pd.options.display.float_format = lambda x: f"{x:.6f}"

print("\n===== Desempenho por etapa/fold (long) =====")
print(df_all.to_string(index=False))

print("\n===== Resumo por etapa (média ± desvio) =====")
summary_cols = [
    "Acurácia","AUROC","AUPRC",
    "Cross-Entropy C0","Proporção C0","Cross-Entropy C1","Proporção C1",
    "F1 Score","Balanced Accuracy","Precision","Recall",
    "Specificity(TNR)","FPR","YoudenJ",
    "TP","FP","TN","FN","TP(%)","FP(%)","TN(%)","FN(%)","Threshold"
]
summary = (df_all
           .groupby(["etapa","modo"])[summary_cols]
           .agg(['mean','std'])
          )
print(summary)



[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
[I 2026-01-26 03:07:07,315] A new study created in memory with name: rf_optuna_8fold_search_catnum


Note: you may need to restart the kernel to use updated packages.


Best trial: 0. Best value: 0.819906:   3%|▎         | 1/30 [00:09<04:46,  9.86s/it]

[I 2026-01-26 03:07:17,179] Trial 0 finished with value: 0.8199062127604171 and parameters: {'n_estimators': 500, 'max_depth_choice': 'None', 'min_samples_split': 13, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'float', 'max_features_float': 0.8659541126403374, 'max_samples': 0.6061695553391381}. Best is trial 0 with value: 0.8199062127604171.


Best trial: 1. Best value: 0.821532:   7%|▋         | 2/30 [00:16<03:36,  7.74s/it]

...

[I 2026-01-26 03:14:23,958] Trial 27 finished with value: 0.8295032601305228 and parameters: {'n_estimators': 800, 'max_depth_choice': 'int', 'max_depth': 15, 'min_samples_split': 14, 'min_samples_leaf': 16, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'sqrt', 'max_samples': 0.9809752147751409}. Best is trial 12 with value: 0.8361046401763449.


Best trial: 12. Best value: 0.836105:  97%|█████████▋| 29/30 [07:35<00:16, 16.60s/it]

[I 2026-01-26 03:14:42,910] Trial 28 finished with value: 0.8318866499429483 and parameters: {'n_estimators': 950, 'max_depth_choice': 'int', 'max_depth': 7, 'min_samples_split': 19, 'min_samples_leaf': 18, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'float', 'max_features_float': 0.2407519562011199, 'max_samples': 0.5507661166321712}. Best is trial 12 with value: 0.8361046401763449.


Best trial: 12. Best value: 0.836105: 100%|██████████| 30/30 [07:46<00:00, 15.55s/it]


[I 2026-01-26 03:14:53,801] Trial 29 finished with value: 0.8254056345579712 and parameters: {'n_estimators': 500, 'max_depth_choice': 'int', 'max_depth': 24, 'min_samples_split': 12, 'min_samples_leaf': 10, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'log2', 'max_samples': 0.6986396182085532}. Best is trial 12 with value: 0.8361046401763449.
Melhor AUROC (CV interna 8 folds): 0.8361046401763449
Melhores hiperparâmetros: {'n_estimators': 1000, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 20, 'bootstrap': True, 'class_weight': None, 'max_samples': 0.5015694891570545, 'max_features': 'sqrt', 'n_jobs': -1, 'random_state': 42}

===== Thresholds 'interessantes' escolhidos no fold 9 =====
                   criterio  threshold
                    best_f1   0.150000
      best_recall_fpr<=0.01   0.173333
      best_recall_fpr<=0.02   0.163434
      best_recall_fpr<=0.03   0.150000
      best_recall_fpr<=0.05   0.133737
                best_youden   0.024848
m

In [ ]:
import numpy as np
import pandas as pd

# ==========================================================
# PLUGIN — LOG FINAL DE MÉTRICAS (VALIDAÇÃO + TESTE)
# ==========================================================
def report_final_metrics(
    df_long,
    threshold_mode="best_f1",
    etapa_validacao="validacao",
    etapa_teste="teste",
    print_table=True
):
    """
    df_long: DataFrame longo com todas as métricas por fold/etapa
    threshold_mode: modo escolhido no fold de validação (ex: 'best_f1')
    """

    METRIC_COLS = [
        "Acurácia", "F1 Score", "Balanced Accuracy",
        "Precision", "Recall", "Specificity(TNR)",
        "AUROC", "AUPRC",
        "Cross-Entropy C0", "Cross-Entropy C1",
        "TP(%)", "FP(%)", "TN(%)", "FN(%)"
    ]

    rows = []

    for etapa in [etapa_validacao, etapa_teste]:
        df_sel = df_long[
            (df_long["etapa"] == etapa) &
            (df_long["modo"] == threshold_mode)
        ].copy()

        thr_mean = df_sel["Threshold"].mean()
        thr_std  = df_sel["Threshold"].std()

        metrics_mean = df_sel[METRIC_COLS].mean()
        metrics_std  = df_sel[METRIC_COLS].std()

        row = {
            "Etapa": etapa,
            "Threshold (mean ± std)": f"{thr_mean:.4f} ± {0 if np.isnan(thr_std) else thr_std:.4f}"
        }

        for m in METRIC_COLS:
            mu = metrics_mean[m]
            sd = metrics_std[m]
            row[m] = f"{mu:.4f} ± {0 if np.isnan(sd) else sd:.4f}"

        rows.append(row)

    df_report = pd.DataFrame(rows)

    if print_table:
        print("\n================ RESULTADO FINAL (30 repetições) ================\n")
        print(f"Threshold escolhido no fold de validação: '{threshold_mode}'\n")
        print(df_report.to_string(index=False))
        print("\n=================================================================\n")

    return df_report


In [ ]:
df_final = report_final_metrics(
    df_long=30,   # <- concat das 30 execuções
    threshold_mode="best_f1"    # <- OU outro que você escolher
)


In [ ]:
rf_params

In [ ]:
# ================== IMPORTS ==================
import os, time, joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, log_loss
)
from sklearn.ensemble import RandomForestClassifier
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm

# ================== PREPARAÇÃO X, y ==================
y = df['y']

# Detecta colunas identificadoras a partir de EXCLUDE_COLUMNS (case-insensitive)
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != 'y']  # garante que não inclua a target

# FEATURES são todas as colunas exceto ids e y
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]
X_full = df[ID_COLS + FEATURES] if ID_COLS else df[FEATURES]

# Detecta tipos (numéricas x categóricas) a partir de FEATURES
num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in FEATURES if c not in num_cols]

# K-Fold estratificado (será usado apenas para obter os índices e formar 8/1/1)
kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_indices = list(kf.split(df[FEATURES], y))

# ================== MONTA 8/1/1 (fixo) ==================
# 10º = teste, 9º = validação, demais (1..8) = treino
train8_idx, val1_idx, test1_idx = None, None, None
for i, (tr_idx, ts_idx) in enumerate(fold_indices, start=1):
    if i == len(fold_indices):          # 10º -> TESTE
        test1_idx = ts_idx
    elif i == len(fold_indices) - 1:    # 9º -> VALIDAÇÃO
        val1_idx = ts_idx
    else:                               # 1..8 -> TREINO (acumula os índices de "ts_idx")
        train8_idx = ts_idx if train8_idx is None else np.concatenate([train8_idx, ts_idx])

# Partições completas (mantendo IDs p/ rastreabilidade)
X_train8_full = X_full.iloc[train8_idx].copy()
X_val1_full   = X_full.iloc[val1_idx].copy()
X_test1_full  = X_full.iloc[test1_idx].copy()

# Apenas as FEATURES para modelagem
X_train8 = X_train8_full[FEATURES].copy()
X_val1   = X_val1_full[FEATURES].copy()
X_test1  = X_test1_full[FEATURES].copy()

y_train8 = y.iloc[train8_idx]
y_val1   = y.iloc[val1_idx]
y_test1  = y.iloc[test1_idx]

# ================== FUNÇÃO DE PRÉ-PROCESSAMENTO POR CONJUNTO (SEM VAZAMENTO) ==================
def _prep_fold(X_tr, X_te):
    # 1) Categóricas → OrdinalEncoder (fit no treino)
    if cat_cols:
        Xtr_cat = X_tr[cat_cols].astype('object').fillna('Unknown')
        Xte_cat = X_te[cat_cols].astype('object').fillna('Unknown')

        oenc = OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,
            encoded_missing_value=-1
        )
        Xtr_cat_enc = pd.DataFrame(oenc.fit_transform(Xtr_cat), columns=cat_cols, index=X_tr.index)
        Xte_cat_enc = pd.DataFrame(oenc.transform(Xte_cat),  columns=cat_cols, index=X_te.index)
    else:
        Xtr_cat_enc = pd.DataFrame(index=X_tr.index)
        Xte_cat_enc = pd.DataFrame(index=X_te.index)

    # 2) Numéricas → coerção + imputação por mediana do TREINO
    if num_cols:
        Xtr_num = X_tr[num_cols].apply(pd.to_numeric, errors='coerce')
        Xte_num = X_te[num_cols].apply(pd.to_numeric, errors='coerce')
        med = Xtr_num.median(numeric_only=True)
        Xtr_num = Xtr_num.fillna(med)
        Xte_num = Xte_num.fillna(med)
    else:
        Xtr_num = pd.DataFrame(index=X_tr.index)
        Xte_num = pd.DataFrame(index=X_te.index)

    # 3) Reconstrói no MESMO ORDEM DE FEATURES
    Xtr_proc = pd.concat([Xtr_num, Xtr_cat_enc], axis=1)[FEATURES]
    Xte_proc = pd.concat([Xte_num, Xte_cat_enc], axis=1)[FEATURES]
    return Xtr_proc, Xte_proc

# ================== ESTRUTURAS DE RESULTADOS ==================
resultados = []
importancias_lista = []

# Estruturas de “partes” (compatíveis com seu pipeline anterior)
X_train_parts, X_test_parts = [], []
y_train_parts, y_test_parts = [], []
y_train_pred_parts, y_test_pred_parts = [], []

start_total = time.time()

# ================== (1) TREINO NOS 8 FOLDS → AVALIA VALIDAÇÃO (fold 9) ==================
t0_val = time.time()
Xtr8_proc, Xva_proc = _prep_fold(X_train8, X_val1)

model_val = RandomForestClassifier(**rf_params)
model_val.fit(Xtr8_proc, y_train8)

# Probabilidades / preds na validação
proba_val_all = model_val.predict_proba(Xva_proc)
proba_val_C1  = proba_val_all[:, 1]
y_pred_val    = (proba_val_C1 >= THRESHOLD).astype(int)

# Métricas de validação (fold 9)
cm_val = confusion_matrix(y_val1, y_pred_val)
tn_v, fp_v, fn_v, tp_v = cm_val.ravel()
total_v = cm_val.sum()

resultados.append({
    'Conjunto': 'Validação(fold9)', 'Fold': 9,
    'Acurácia': accuracy_score(y_val1, y_pred_val),
    'Cross-Entropy C0': log_loss(y_val1[y_val1==0], proba_val_C1[y_val1==0], labels=[0,1]) if (y_val1==0).sum()>0 else np.nan,
    'Proporção C0': y_val1.value_counts(normalize=True).get(0, np.nan) * 100,
    'Cross-Entropy C1': log_loss(y_val1[y_val1==1], proba_val_C1[y_val1==1], labels=[0,1]) if (y_val1==1).sum()>0 else np.nan,
    'Proporção C1': y_val1.value_counts(normalize=True).get(1, np.nan) * 100,
    'F1 Score': f1_score(y_val1, y_pred_val),
    'Balanced Accuracy': balanced_accuracy_score(y_val1, y_pred_val),
    'Precision': precision_score(y_val1, y_pred_val),
    'Recall': recall_score(y_val1, y_pred_val),
    'AUROC': roc_auc_score(y_val1, proba_val_C1),
    'TP': tp_v, 'FP': fp_v, 'TN': tn_v, 'FN': fn_v,
    'TP(%)': round(tp_v/total_v*100, 2), 'FP(%)': round(fp_v/total_v*100, 2),
    'TN(%)': round(tn_v/total_v*100, 2), 'FN(%)': round(fn_v/total_v*100, 2),
    'Tempo (s)': round(time.time() - t0_val, 2)
})

# Blocos de validação (se quiser rastrear como nos K-Folds; marca fold=9)
X_val_block  = X_val1_full.assign(fold=9).reset_index()
y_val_block  = pd.Series(y_val1, name='y_test').to_frame().assign(fold=9).reset_index()
probs_val_df = pd.DataFrame(proba_val_all, columns=['prob_0','prob_1'], index=X_val1_full.index).assign(fold=9).reset_index()

# ================== (2) TREINO FINAL (8+1 = 9 FOLDS) → TESTE (fold 10) ==================
X_train9_full = pd.concat([X_train8_full, X_val1_full])
y_train9      = pd.concat([y_train8, y_val1])

X_train9 = X_train9_full[FEATURES].copy()
Xtr9_proc, Xts_proc = _prep_fold(X_train9, X_test1)

t0_fit = time.time()
model_final = RandomForestClassifier(**rf_params)
model_final.fit(Xtr9_proc, y_train9)
fit_time = time.time() - t0_fit

# Probabilidades e classes
proba_test_all = model_final.predict_proba(Xts_proc)
proba_test_C1  = proba_test_all[:, 1]
y_pred_test    = (proba_test_C1 >= THRESHOLD).astype(int)

# (Opcional) métricas no treino 9 folds para referência
proba_train_all = model_final.predict_proba(Xtr9_proc)
proba_train_C1  = proba_train_all[:, 1]
y_pred_train    = (proba_train_C1 >= THRESHOLD).astype(int)

cm_tr = confusion_matrix(y_train9, y_pred_train)
tn_tr, fp_tr, fn_tr, tp_tr = cm_tr.ravel()
total_tr = cm_tr.sum()

resultados.append({
    'Conjunto': 'Treino(9folds)', 'Fold': 9,
    'Acurácia': accuracy_score(y_train9, y_pred_train),
    'Cross-Entropy C0': log_loss(y_train9[y_train9==0], proba_train_C1[y_train9==0], labels=[0,1]) if (y_train9==0).sum()>0 else np.nan,
    'Proporção C0': y_train9.value_counts(normalize=True).get(0, np.nan) * 100,
    'Cross-Entropy C1': log_loss(y_train9[y_train9==1], proba_train_C1[y_train9==1], labels=[0,1]) if (y_train9==1).sum()>0 else np.nan,
    'Proporção C1': y_train9.value_counts(normalize=True).get(1, np.nan) * 100,
    'F1 Score': f1_score(y_train9, y_pred_train),
    'Balanced Accuracy': balanced_accuracy_score(y_train9, y_pred_train),
    'Precision': precision_score(y_train9, y_pred_train),
    'Recall': recall_score(y_train9, y_pred_train),
    'AUROC': roc_auc_score(y_train9, proba_train_C1),
    'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
    'TP(%)': round(tp_tr/total_tr*100, 2), 'FP(%)': round(fp_tr/total_tr*100, 2),
    'TN(%)': round(tn_tr/total_tr*100, 2), 'FN(%)': round(fn_tr/total_tr*100, 2),
    'Tempo (s)': round(fit_time, 2)
})

# Métricas de teste (fold 10)
cm_ts = confusion_matrix(y_test1, y_pred_test)
tn_ts, fp_ts, fn_ts, tp_ts = cm_ts.ravel()
total_ts = cm_ts.sum()

resultados.append({
    'Conjunto': 'Teste(fold10)', 'Fold': 10,
    'Acurácia': accuracy_score(y_test1, y_pred_test),
    'Cross-Entropy C0': log_loss(y_test1[y_test1==0], proba_test_C1[y_test1==0], labels=[0,1]) if (y_test1==0).sum()>0 else np.nan,
    'Proporção C0': y_test1.value_counts(normalize=True).get(0, np.nan) * 100,
    'Cross-Entropy C1': log_loss(y_test1[y_test1==1], proba_test_C1[y_test1==1], labels=[0,1]) if (y_test1==1).sum()>0 else np.nan,
    'Proporção C1': y_test1.value_counts(normalize=True).get(1, np.nan) * 100,
    'F1 Score': f1_score(y_test1, y_pred_test),
    'Balanced Accuracy': balanced_accuracy_score(y_test1, y_pred_test),
    'Precision': precision_score(y_test1, y_pred_test),
    'Recall': recall_score(y_test1, y_pred_test),
    'AUROC': roc_auc_score(y_test1, proba_test_C1),
    'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
    'TP(%)': round(tp_ts/total_ts*100, 2), 'FP(%)': round(fp_ts/total_ts*100, 2),
    'TN(%)': round(tn_ts/total_ts*100, 2), 'FN(%)': round(fn_ts/total_ts*100, 2),
    'Tempo (s)': round(time.time() - t0_fit, 2)
})

# --------- BLOCOS (compatibilidade com pipeline) ---------
# Validação (fold 9)
X_test_parts.append(X_val_block)      # “teste” = validação aqui, só para manter compatibilidade
y_test_parts.append(y_val_block)
y_test_pred_parts.append(probs_val_df)

# Treino 9 folds (fold 9) – salvamos como “train”
X_train_block = X_train9_full.assign(fold=9).reset_index()
y_train_block = pd.Series(y_train9, name='y_train').to_frame().assign(fold=9).reset_index()
probs_train_df = pd.DataFrame(proba_train_all, columns=['prob_0','prob_1'], index=X_train9_full.index).assign(fold=9).reset_index()

X_train_parts.append(X_train_block)
y_train_parts.append(y_train_block)
y_train_pred_parts.append(probs_train_df)

# Teste (fold 10)
X_test_block = X_test1_full.assign(fold=10).reset_index()
y_test_block = pd.Series(y_test1, name='y_test').to_frame().assign(fold=10).reset_index()
probs_test_df = pd.DataFrame(proba_test_all, columns=['prob_0','prob_1'], index=X_test1_full.index).assign(fold=10).reset_index()

X_test_parts.append(X_test_block)
y_test_parts.append(y_test_block)
y_test_pred_parts.append(probs_test_df)

# --------- IMPORTÂNCIA DAS FEATURES (final, conjunto de teste) ---------
imp_vals = getattr(model_final, "feature_importances_", None)
if imp_vals is not None:
    linha_importancia = {'Conjunto': 'Teste(fold10)', 'Fold': 10}
    for nome, valor in zip(FEATURES, imp_vals):
        linha_importancia[nome] = f"{valor * 100:.2f}%"
    importancias_lista.append(linha_importancia)

# Salva modelo final (mantendo padrão de nome)
model_path = MODEL_DIR / f'rf_model_cf{N_SPLITS}.pkl'
joblib.dump(model_final, model_path)

# ================== PÓS-BLOCOS: SALVAMENTOS ==================
df_resultados = pd.DataFrame(resultados)

# adiciona linha de média das métricas numéricas
mean_row = df_resultados.select_dtypes(include=[np.number]).mean()
mean_row['Conjunto'] = 'Média'
mean_row['Fold'] = 'Média'
df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# salva csvs principais
CSV_OUT.mkdir(parents=True, exist_ok=True)
df_resultados.to_csv(CSV_OUT / f'rf_cv_results_{N_SPLITS}.csv', index=False)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'rf_feature_importances_{N_SPLITS}.csv', index=False)

# Concatena blocos para gerar dataframes completos de treino/teste com probabilidades
X_train_global = pd.concat(X_train_parts, ignore_index=True) if X_train_parts else pd.DataFrame()
y_train_global = pd.concat(y_train_parts, ignore_index=True) if y_train_parts else pd.DataFrame()
y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True) if y_train_pred_parts else pd.DataFrame()

X_test_global = pd.concat(X_test_parts, ignore_index=True) if X_test_parts else pd.DataFrame()
y_test_global = pd.concat(y_test_parts, ignore_index=True) if y_test_parts else pd.DataFrame()
y_test_pred_global = pd.concat(y_test_pred_parts, ignore_index=True) if y_test_pred_parts else pd.DataFrame()

# Renomeia 'index' -> 'orig_index' (índice original)
for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
    if not df_block.empty and 'index' in df_block.columns:
        df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# Mesclas finais (mantém nomes/formatos anteriores)
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
else:
    train_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
else:
    test_df = pd.DataFrame()

# Adiciona y_proba / y_pred e salva
if not test_df.empty:
    test_df['y_proba'] = test_df['prob_1']
    test_df['y_pred']  = (test_df['y_proba'] >= THRESHOLD).astype(int)
    test_df.to_csv(CSV_OUT / f'rf_test_with_proba_{N_SPLITS}.csv', index=False)

if not train_df.empty:
    train_df['y_proba'] = train_df['prob_1']
    train_df['y_pred']  = (train_df['y_proba'] >= THRESHOLD).astype(int)
    train_df.to_csv(CSV_OUT / f'rf_train_with_proba_{N_SPLITS}.csv', index=False)

# Dataset unificado (opcional)
if not test_df.empty:
    Path('../data/processed').mkdir(parents=True, exist_ok=True)
    test_df.to_csv(Path('../data/processed') / f'stroke_rf_test_with_proba_{N_SPLITS}.csv', index=False)

# Augmenta o df original com previsões do TESTE (por orig_index)
try:
    if not test_df.empty:
        df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
        preds = test_df[['orig_index', 'y_pred', 'y_proba']]
        df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
        df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_{N_SPLITS}.csv', index=False)
    else:
        print('test_df vazio: não foi possível gerar dataset augmentado com previsões')
except Exception as e_aug:
    print('Erro ao gerar dataset augmentado com previsões:', e_aug)

print('Resultados salvos em:', CSV_OUT)


In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
# Necessário para auc(fpr, tpr) e para garantir imports mínimos locais
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import auc  # necessário para usar auc(fpr, tpr)

# Geração de PDF resumo (com algumas páginas importantes)
with PdfPages(PDF_OUT / f'RF_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # ---------------- Página de parâmetros ----------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = rf_params.copy()
    parametros['threshold'] = THRESHOLD
    parametros['n_splits'] = N_SPLITS
    texto = 'Algoritmo: Random Forest\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', verticalalignment='top')
    ax.set_title('📌 Parâmetros do Modelo - Random Forest')
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig)
    plt.close()

    # ---------------- Página de métricas resumidas ----------------
    if not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = df_resultados.groupby(['Conjunto']).mean(numeric_only=True).T.round(4)
        table = ax.table(cellText=display_df.values,
                         colLabels=display_df.columns,
                         rowLabels=display_df.index,
                         loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig)
        plt.close()

    # ---------------- Página ROC - Treino vs Teste (lado a lado) ----------------
    try:
        plotted = False

        # ROC Treino (usa dado consolidado de 9 folds de treino)
        if 'y_train' in train_df.columns and 'prob_1' in train_df.columns:
            y_train_all = train_df['y_train']
            y_score_train_all = train_df['prob_1']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train)
            plotted = True
        elif len(fprs_train) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_train, tprs_train):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if len(aurocs_train) > 0 else auc(fpr_train, tpr_train)
            plotted = True

        # ROC Teste (AQUI AJUSTE 8/1/1: usa SOMENTE fold 10)
        if 'y_test' in test_df.columns and 'prob_1' in test_df.columns:
            test_block = test_df
            if 'fold' in test_block.columns:
                test_block = test_block[test_block['fold'] == 10]  # <- filtra o fold de TESTE
            y_test_all = test_block['y_test']
            y_score_test_all = test_block['prob_1']
            if len(y_test_all) > 0:
                fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
                roc_auc_test = auc(fpr_test, tpr_test)
                plotted = True
        elif len(fprs_test) > 0:
            # Se tiver armazenado curvas por fold, você pode filtrar apenas as de fold 10 aqui
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_test, tprs_test):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if len(aurocs_test) > 0 else auc(fpr_test, tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            # ROC Treino
            try:
                axes[0].plot(fpr_train, tpr_train, label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[0].set_title('ROC - Treino (9 folds)')
                axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5, 0.5, 'Não há dados de treino para ROC', ha='center')
            # ROC Teste (fold 10)
            try:
                axes[1].plot(fpr_test, tpr_test, label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[1].set_title('ROC - Teste (fold 10)')
                axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5, 0.5, 'Não há dados de teste para ROC', ha='center')
            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # ---------------- Página Confusion Matrix - Treino vs Teste ----------------
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            y_true = df_block[true_col]
            y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum()
            cm_percent = cm / total * 100 if total > 0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        cm_train = cm_train_pct = cm_test = cm_test_pct = None

        # Treino (9 folds)
        if 'y_train' in train_df.columns and 'y_pred' in train_df.columns:
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        else:
            if 'TP' in df_resultados.columns:
                train_rows = df_resultados[df_resultados['Conjunto'].str.startswith('Treino')]
                TP = int(train_rows['TP'].sum()) if not train_rows.empty else 0
                FP = int(train_rows['FP'].sum()) if not train_rows.empty else 0
                TN = int(train_rows['TN'].sum()) if not train_rows.empty else 0
                FN = int(train_rows['FN'].sum()) if not train_rows.empty else 0
                cm_train = np.array([[TN, FP], [FN, TP]])
                total = cm_train.sum()
                cm_train_pct = cm_train / total * 100 if total > 0 else np.zeros_like(cm_train, dtype=float)

        # Teste (fold 10) — AJUSTE 8/1/1: filtra o fold 10
        if 'y_test' in test_df.columns and 'y_pred' in test_df.columns:
            test_block = test_df
            if 'fold' in test_block.columns:
                test_block = test_block[test_block['fold'] == 10]  # <- apenas fold 10
            if len(test_block) > 0:
                cm_test, cm_test_pct = build_cm_from_df(test_block, 'y_test', 'y_pred')
        else:
            if 'TP' in df_resultados.columns:
                test_rows = df_resultados[df_resultados['Conjunto'].str.startswith('Teste')]
                TP = int(test_rows['TP'].sum()) if not test_rows.empty else 0
                FP = int(test_rows['FP'].sum()) if not test_rows.empty else 0
                TN = int(test_rows['TN'].sum()) if not test_rows.empty else 0
                FN = int(test_rows['FN'].sum()) if not test_rows.empty else 0
                cm_test = np.array([[TN, FP], [FN, TP]])
                total = cm_test.sum()
                cm_test_pct = cm_test / total * 100 if total > 0 else np.zeros_like(cm_test, dtype=float)

        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center'); ax.axis('off'); return
                annot = np.empty(cm.shape, dtype=object)
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]):
                        annot[i, j] = f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)"
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1']); ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino (9 folds)')
            plot_cm(axes[1], cm_test, cm_test_pct, 'Matriz de Confusão - Teste (fold 10)')

            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # ---------------- Importâncias médias + Pareto ----------------
    try:
        df_imp = pd.DataFrame(importancias_lista).replace('%', '', regex=True)
        cols = [c for c in df_imp.columns if c not in ['Conjunto', 'Fold']]
        try:
            exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
        except Exception:
            exclude_lower = set()
        cols = [c for c in cols if c.lower() not in exclude_lower]

        if len(cols) == 0:
            print('Nenhuma feature disponível para plotar importâncias (após excluir ids).')
        else:
            df_imp_mean = df_imp[cols].astype(float).mean().sort_values(ascending=True)

            height = max(6, 0.25 * len(df_imp_mean))
            fig, ax = plt.subplots(figsize=(10, height))
            df_imp_mean.plot(kind='barh', ax=ax, color='tab:blue')
            ax.set_xlabel('Importância média (%)')
            ax.set_title('Importância média das features (porcentagem)')
            plt.tight_layout(); plt.subplots_adjust(left=0.30)
            try:
                png_path = IMAGES_OUT / f'feature_importances_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()

            # Tabela de importâncias
            try:
                table_df = df_imp_mean.sort_values(ascending=False).to_frame(name='Importance (%)').round(2)
                table_height = max(4, 0.3 * len(table_df))
                fig, ax = plt.subplots(figsize=(8, table_height))
                ax.axis('off')
                table = ax.table(cellText=table_df.values,
                                 colLabels=table_df.columns,
                                 rowLabels=table_df.index,
                                 loc='center')
                table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, 1.2)
                ax.set_title('Tabela: Importância média das features (%)', pad=20)
                plt.tight_layout()
                pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_table:
                print('Não foi possível gerar tabela de importâncias:', e_table)

            # --- Pareto 80/20 ---
            try:
                imp_desc = df_imp_mean.sort_values(ascending=False)
                pareto_df = imp_desc.to_frame(name='importance_pct')
                pareto_df['cumulative_pct'] = pareto_df['importance_pct'].cumsum()
                threshold_pct = (PARETO_THRESHOLD if 'PARETO_THRESHOLD' in globals() else 0.8) * 100

                import numpy as _np
                pos = _np.searchsorted(pareto_df['cumulative_pct'].values, threshold_pct)
                if pos < len(pareto_df):
                    selected_features = pareto_df.iloc[:pos+1].index.tolist()
                else:
                    selected_features = pareto_df.index.tolist()

                PARETO_SELECTED_FEATURES = selected_features
                PARETO_SELECTED_IMPORTANCES = imp_desc.loc[selected_features].round(2).to_dict()

                pareto_df.reset_index().rename(columns={'index': 'feature'}) \
                         .to_csv(CSV_OUT / f'pareto_importances_{int((PARETO_THRESHOLD*100))}.csv', index=False)
                pd.DataFrame({'feature': PARETO_SELECTED_FEATURES,
                              'importance_pct': [PARETO_SELECTED_IMPORTANCES[f] for f in PARETO_SELECTED_FEATURES]}) \
                  .to_csv(CSV_OUT / f'pareto_selected_features_{int((PARETO_THRESHOLD*100))}.csv', index=False)

                # Gráfico Pareto
                try:
                    fig, ax1 = plt.subplots(figsize=(10, 6))
                    x = pareto_df.index.astype(str)
                    ax1.bar(x, pareto_df['importance_pct'], color='tab:blue')
                    ax1.set_ylabel('Importância (%)')
                    ax1.set_xticklabels(x, rotation=90)
                    ax2 = ax1.twinx()
                    ax2.plot(x, pareto_df['cumulative_pct'], color='red', marker='o')
                    ax2.axhline(threshold_pct, color='gray', linestyle='--')
                    ax2.set_ylabel('Cumulative (%)')
                    ax1.set_title(f'Pareto (threshold={int(threshold_pct)}%) - selecionadas: {len(PARETO_SELECTED_FEATURES)}')
                    plt.tight_layout()
                    try:
                        png_path = IMAGES_OUT / f'pareto_{int((PARETO_THRESHOLD*100))}.png'
                        fig.savefig(png_path, dpi=300, bbox_inches='tight')
                    except Exception:
                        pass
                    pdf.savefig(fig, bbox_inches='tight'); plt.close()
                except Exception as e_plot:
                    print('Não foi possível gerar gráfico Pareto:', e_plot)

            except Exception as e_pareto:
                print('Não foi possível gerar análise Pareto:', e_pareto)

            # --- Tabela de variâncias: Treino vs Teste (APENAS NUMÉRICAS) ---
            try:
                if 'FEATURES' in globals() and not (train_df.empty or test_df.empty):
                    feats_common = [f for f in FEATURES if f in train_df.columns and f in test_df.columns]
                    if len(feats_common) == 0:
                        print('Nenhuma feature comum entre train/test para calcular variância.')
                    else:
                        # Filtra teste para fold 10 (8/1/1)
                        test_block = test_df
                        if 'fold' in test_block.columns:
                            test_block = test_block[test_block['fold'] == 10]

                        # Converte para numérico
                        train_num = train_df[feats_common].apply(pd.to_numeric, errors='coerce')
                        test_num  = test_block[feats_common].apply(pd.to_numeric, errors='coerce')

                        # mantém apenas colunas com pelo menos 1 valor numérico em cada
                        valid_cols = [c for c in feats_common if train_num[c].notna().any() and test_num[c].notna().any()]
                        if len(valid_cols) == 0:
                            print('Nenhuma feature numérica válida em comum para variância.')
                        else:
                            var_train = train_num[valid_cols].var(ddof=1, numeric_only=True)
                            var_test  = test_num[valid_cols].var(ddof=1, numeric_only=True)
                            var_df = pd.DataFrame({'var_train': var_train, 'var_test': var_test})
                            var_df['diff'] = var_df['var_train'] - var_df['var_test']
                            var_df['pct_diff'] = (var_df['diff'].abs() / var_df['var_train'].replace({0: np.nan})) * 100

                            try:
                                thresh_pct = VAR_DIFF_PCT_THRESHOLD * 100
                            except Exception:
                                thresh_pct = 10.0
                            var_df['high_low'] = var_df['pct_diff'].apply(
                                lambda x: 'High' if (pd.notna(x) and x >= thresh_pct) else 'Low'
                            )
                            var_df = var_df.sort_values('var_train', ascending=False)

                            out_var_csv = CSV_OUT / 'feature_variances_train_test.csv'
                            var_df.reset_index().rename(columns={'index': 'feature'}).to_csv(out_var_csv, index=False)

                            # Página com a tabela
                            try:
                                max_rows = len(var_df)
                                fig_h = min(12, 0.25 * max_rows + 2)
                                fig, ax = plt.subplots(figsize=(11.69, fig_h))
                                ax.axis('off')
                                tbl = ax.table(
                                    cellText=var_df.round(4).reset_index().values,
                                    colLabels=['feature', 'var_train', 'var_test', 'diff', 'pct_diff', 'high_low'],
                                    loc='center'
                                )
                                tbl.auto_set_font_size(False)
                                tbl.set_fontsize(8)
                                tbl.scale(1, 1.2)
                                ax.set_title('Tabela: Variância das Features (Treino vs Teste - fold 10)')
                                plt.tight_layout()
                                pdf.savefig(fig, bbox_inches='tight'); plt.close()
                            except Exception as e_tbl:
                                print('Não foi possível gerar página PDF da tabela de variâncias:', e_tbl)
                else:
                    print('Dados de treino/teste ou FEATURES ausentes; não foi possível calcular variâncias.')
            except Exception as e_var:
                print('Erro ao calcular tabela de variâncias:', e_var)

            # --- Gráficos de violino (usa TESTE apenas fold 10) ---
            try:
                if (PARETO_SELECTED_FEATURES is None) or (len(PARETO_SELECTED_FEATURES) == 0):
                    print('Nenhuma feature selecionada pelo Pareto para gerar violinos.')
                elif train_df.empty or test_df.empty:
                    print('Dados de treino ou teste ausentes para gerar violinos.')
                else:
                    violin_features = [f for f in PARETO_SELECTED_FEATURES if f in train_df.columns and f in test_df.columns]
                    if len(violin_features) == 0:
                        print('Nenhuma das features selecionadas pelo Pareto está presente em train/test para plotagem.')
                    else:
                        test_block = test_df
                        if 'fold' in test_block.columns:
                            test_block = test_block[test_block['fold'] == 10]  # <- somente teste real

                        combined = pd.concat([
                            train_df[violin_features].copy().assign(__set__='train'),
                            test_block[violin_features].copy().assign(__set__='test')
                        ], ignore_index=True)

                        per_page = 6
                        import math
                        n = len(violin_features)
                        pages = math.ceil(n / per_page)
                        for p in range(pages):
                            start = p * per_page
                            end = min(start + per_page, n)
                            page_feats = violin_features[start:end]
                            rows = 3
                            fig, axes = plt.subplots(rows, 2, figsize=(8.27, 11.69))  # A4 portrait
                            axes = axes.flatten()
                            for i, feat in enumerate(page_feats):
                                ax = axes[i]
                                sns.violinplot(x='__set__', y=feat, data=combined,
                                               order=['train', 'test'], ax=ax,
                                               inner='quartile', palette=['#4c72b0', '#55a868'])
                                ax.set_title(f'Violin: {feat} (train vs test)')
                                ax.set_xlabel('')
                            for j in range(len(page_feats), len(axes)):
                                axes[j].axis('off')
                            plt.tight_layout()
                            pdf.savefig(fig, bbox_inches='tight')

                            # imagens individuais
                            try:
                                for feat in page_feats:
                                    fig_f, ax_f = plt.subplots(figsize=(6, 4))
                                    sns.violinplot(x='__set__', y=feat, data=combined,
                                                   order=['train', 'test'], ax=ax_f,
                                                   inner='quartile', palette=['#4c72b0', '#55a868'])
                                    ax_f.set_title(f'Violin: {feat} (train vs test)')
                                    ax_f.set_xlabel('')
                                    plt.tight_layout()
                                    png_path = IMAGES_OUT / f'violin_{feat}.png'
                                    fig_f.savefig(png_path, dpi=300, bbox_inches='tight')
                                    plt.close(fig_f)
                            except Exception:
                                pass
                            plt.close(fig)
            except Exception as e_violin:
                print('Não foi possível gerar gráficos de violino:', e_violin)
    except Exception as e:
        print('Não foi possível gerar gráfico de importâncias:', e)

print('PDF resumo gerado em:', PDF_OUT)


In [ ]:
# ==========================================================
# Salvar X_test_basic_full.csv e X_train_basic_full.csv
# (8/1/1): teste = fold 10; treino = folds 1..9 (8+1)
# ==========================================================
import pandas as pd
import numpy as np
from pathlib import Path

# Diretório de saída (mantido)
BASICO_CSV = Path("../model_reports/random_forest/basico/csv")
BASICO_CSV.mkdir(parents=True, exist_ok=True)

# ---- Carrega o arquivo cru e indexa por orig_index p/ .loc consistente ---
raw_df = pd.read_csv(DATASET_PATH)
if 'orig_index' not in raw_df.columns:
    # preserva o índice original como 'orig_index' e usa como índice do DF
    raw_df = raw_df.reset_index().rename(columns={'index': 'orig_index'}).set_index('orig_index')
else:
    # garante que seja o índice
    raw_df = raw_df.set_index('orig_index')

# Se existir a coluna 'stroke', criar 'y' (mantido)
if 'stroke' in raw_df.columns and 'y' not in raw_df.columns:
    raw_df['y'] = raw_df['stroke']

# ---------------- Função auxiliar de salvamento ----------------
def save_full_split(indices, y_true, y_pred, y_proba, split_name):
    # garante 1D alinhado por indices
    y_true_s  = pd.Series(np.asarray(y_true),  index=indices, name='y')
    y_pred_s  = pd.Series(np.asarray(y_pred),  index=indices, name='y_pred')
    y_proba_s = pd.Series(np.asarray(y_proba), index=indices, name='y_proba')

    df_full = raw_df.loc[indices].copy()
    df_full = df_full.assign(y=y_true_s, y_pred=y_pred_s, y_proba=y_proba_s)

    out_path = BASICO_CSV / f"X_{split_name}_basic_full.csv"
    df_full.to_csv(out_path, index=True)
    print(f"✅ Arquivo salvo: {out_path} ({len(df_full)} linhas)")

# ---------------- Seleção 8/1/1 a partir de train_df/test_df ----------------
def _build_from_block(df_block, true_col, proba_col='prob_1'):
    """
    Retorna (indices, y_true, y_pred, y_proba) a partir de um bloco consolidado
    com colunas: 'orig_index', true_col ('y_train'/'y_test'), e prob_1 ou y_proba.
    """
    if df_block.empty:
        return [], [], [], []

    # Índices originais
    indices = df_block['orig_index'].to_numpy()

    # y_true
    y_true = df_block[true_col].to_numpy()

    # Probabilidade positiva: tenta 'y_proba', senão 'prob_1'
    if 'y_proba' in df_block.columns:
        y_proba = df_block['y_proba'].to_numpy()
    elif proba_col in df_block.columns:
        y_proba = df_block[proba_col].to_numpy()
    else:
        raise RuntimeError("Coluna de probabilidade não encontrada em df_block (esperado 'y_proba' ou 'prob_1').")

    # y_pred (se não existir, calcula com THRESHOLD)
    if 'y_pred' in df_block.columns:
        y_pred = df_block['y_pred'].to_numpy()
    else:
        y_pred = (y_proba >= THRESHOLD).astype(int)

    return indices, y_true, y_pred, y_proba

# ---------------- Caminho principal (8/1/1) ----------------
try:
    # TESTE: apenas fold 10
    if 'fold' in test_df.columns:
        test_block = test_df[test_df['fold'] == 10].copy()
    else:
        test_block = test_df.copy()

    tst_idx, tst_y, tst_pred, tst_proba = _build_from_block(
        test_block, true_col='y_test', proba_col='prob_1'
    )
    if len(tst_idx) > 0:
        save_full_split(
            indices=tst_idx,
            y_true=tst_y,
            y_pred=tst_pred,
            y_proba=tst_proba,
            split_name="test"
        )
    else:
        print("⚠️ test_df vazio (ou sem fold 10). Indo para fallback do TESTE.")

except NameError:
    # test_df não existe — fallback mais antigo (apenas se variáveis unitárias existirem)
    try:
        save_full_split(
            indices=X_test_full.index,
            y_true=y_test,
            y_pred=y_pred_test_bin,
            y_proba=y_proba_test_C1,
            split_name="test"
        )
    except Exception as e:
        print("⚠️ Não foi possível salvar TESTE (fallback):", e)

try:
    # TREINO: folds 1..9 (8+1). Se tiver 'fold', exclui 10.
    if 'fold' in train_df.columns:
        train_block = train_df.copy()  # aqui o train_df já agrega os 9 folds
    else:
        train_block = train_df.copy()

    tr_idx, tr_y, tr_pred, tr_proba = _build_from_block(
        train_block, true_col='y_train', proba_col='prob_1'
    )
    if len(tr_idx) > 0:
        save_full_split(
            indices=tr_idx,
            y_true=tr_y,
            y_pred=tr_pred,
            y_proba=tr_proba,
            split_name="train"
        )
    else:
        print("⚠️ train_df vazio. Indo para fallback do TREINO.")

except NameError:
    # train_df não existe — fallback mais antigo (apenas se variáveis unitárias existirem)
    try:
        save_full_split(
            indices=X_train_full.index,
            y_true=y_train,
            y_pred=y_pred_train_bin,
            y_proba=y_proba_train_C1,
            split_name="train"
        )
    except Exception as e:
        print("⚠️ Não foi possível salvar TREINO (fallback):", e)


In [ ]:
import pandas as pd
from pathlib import Path

# Caminhos
BASE_DIR = Path("../STROKE/data/processed")
DATA_PATH = BASE_DIR / "healthcare_stroke_data.csv"
IDX_PATH  = BASE_DIR / "entradas_selecionadas_final.csv"
OUT_PATH  = BASE_DIR / "entradas_selecionadas_final_id.csv"

# 1) Carrega o dataset completo
df_data = pd.read_csv(DATA_PATH)

# 2) Carrega os índices globais (posicionais)
df_idx = pd.read_csv(IDX_PATH)
idx_global = df_idx["idx_global"].to_numpy(dtype=int)

# 3) Seleciona as linhas pela POSIÇÃO (iloc)
df_sel = df_data.iloc[idx_global].copy()

# 4) (Opcional) adicionar a coluna idx_global e o índice real (loc)
df_sel.insert(0, "idx_global", idx_global)          # posição no DataFrame
df_sel.insert(1, "idx_real", df_sel.index.values)   # índice real (loc)

# 5) Salva o CSV final
df_sel.to_csv(OUT_PATH, index=False)

print(f"Arquivo salvo em: {OUT_PATH}")
